# Lab 00: Introduction to NumPy — Sales Data Analysis

This notebook covers four tasks (slides 17–20): simulating sales data, computing monthly averages, comparing loop and vectorized performance, and aggregating sales by week.

## Task 1 — Simulating Sales Data (Slide 17)

I simulate daily sales (in units) for **4 products over 364 days** (52 weeks × 7 days). `np.random.randint` draws integers from 10 to 100 (the upper bound 101 is exclusive). The seed is fixed to 42 so the results are reproducible.

In [ ]:
import numpy as np

# Set a seed for reproducibility
np.random.seed(42)

# Parameters
num_days = 364
num_products = 4

# Simulate daily sales units
daily_sales = np.random.randint(
    low=10,
    high=101,
    size=(num_days, num_products)
)

# Display shape and first 5 days
print("Shape:", daily_sales.shape)
print("First 5 days:\n", daily_sales[:5])

**Explanation.** `daily_sales` is a 2-D array of shape `(364, 4)`: each row is one day, each column is one product. Every value lies between 10 and 100 units. Because of the seed, re-running the notebook gives identical numbers.

## Task 2 — Computing Monthly Averages (Slide 18)

To work with months I use a simplified calendar of **12 months × 30 days = 360 days**. I take the first 360 days of `daily_sales` (so the data stays consistent with Task 1) and reshape it from `(360, 4)` to `(12, 30, 4)`, i.e. `(months, days_per_month, products)`. Averaging over `axis=1` (the days axis) collapses it to `(12, 4)`.

In [ ]:
# Parameters
num_days_month = 360  # 12 months of 30 days

# Use the first 360 days of the simulated data
sales_360 = daily_sales[:num_days_month]
print("Original shape:", sales_360.shape)

# Reshape into (months, days_per_month, products)
monthly_sales = sales_360.reshape(12, 30, 4)
print("Reshaped:", monthly_sales.shape)

# Calculate average daily sales per product per month
monthly_avg_sales = monthly_sales.mean(axis=1)
print("Monthly averages shape:", monthly_avg_sales.shape)
print("Monthly average daily sales (rows = months, columns = products):\n", monthly_avg_sales.round(2))

In [ ]:
# Extra analysis: best and worst month for each product
product_names = [f"Product {i+1}" for i in range(4)]
for j, name in enumerate(product_names):
    best = monthly_avg_sales[:, j].argmax() + 1
    worst = monthly_avg_sales[:, j].argmin() + 1
    print(f"{name}: best month = {best} ({monthly_avg_sales[best-1, j]:.2f}), "
          f"worst month = {worst} ({monthly_avg_sales[worst-1, j]:.2f})")

print("\nOverall average daily sales per product:", monthly_avg_sales.mean(axis=0).round(2))

**Explanation.** Reshaping works here because 360 = 12 × 30, and the order `(months, days, products)` keeps each month's 30 days together. `mean(axis=1)` averages along the days axis, giving 12 rows (months) × 4 columns (products). Most monthly averages fall between about 45 and 62 units, centred near 55, which is the expected mean of a uniform distribution between 10 and 100 (the lowest value, about 38, is just random variation). Differences between months and products are random noise, not real trends.

## Task 3 — Performance Comparison (Slide 19)

I compare summing all sales with **nested Python loops** against NumPy's vectorized **`np.sum`**, using the `daily_sales` array from Task 1.

In [ ]:
def sum_with_loop(data):
    total = 0
    for row in data:
        for item in row:
            total += item
    return total

# Both methods must give the same answer
print("Loop:", sum_with_loop(daily_sales), "| NumPy:", np.sum(daily_sales))

In [ ]:
# Timing with loops
%timeit sum_with_loop(daily_sales)

In [ ]:
# Using NumPy's vectorized sum
%timeit np.sum(daily_sales)

In [ ]:
# Speed-up factor measured with the timeit module
import timeit

t_loop = timeit.timeit(lambda: sum_with_loop(daily_sales), number=100) / 100
t_np = timeit.timeit(lambda: np.sum(daily_sales), number=1000) / 1000
print(f"Loop: {t_loop*1e6:.1f} µs | NumPy: {t_np*1e6:.1f} µs | NumPy is ~{t_loop/t_np:.0f}x faster")

**Explanation.** Both approaches return the same total, so the difference is only speed. The loop runs in the Python interpreter and handles one element at a time with type checks on every step. `np.sum` delegates the iteration to optimized C code operating on a contiguous block of memory. The last cell reports the measured speed-up on this machine; it grows with array size, which is why vectorization is essential for data analysis (the slide quotes 50–100× for larger workloads).

## Task 4 — Weekly Sales Aggregation (Slide 20)

The year has 364 days = 52 weeks × 7 days, so the `(364, 4)` array can be reshaped into `(52, 7, 4)` = `(weeks, days_per_week, products)`. Summing over `axis=1` (the days axis) gives total weekly sales per product, shape `(52, 4)`.

In [ ]:
# daily_sales shape is (364, 4)
# 364 days = 52 weeks * 7 days

# Reshape into (weeks, days_per_week, products)
weekly_sales_data = daily_sales.reshape(52, 7, 4)

# Sum across 'days' axis to get weekly totals
total_weekly_sales = weekly_sales_data.sum(axis=1)

print("Shape of total_weekly_sales:", total_weekly_sales.shape)
print("Total sales for the first week:")
print(total_weekly_sales[0])

In [ ]:
# Extra analysis and sanity check
print("Best week per product (1-based):", total_weekly_sales.argmax(axis=0) + 1)
print("Worst week per product (1-based):", total_weekly_sales.argmin(axis=0) + 1)
print("Average weekly sales per product:", total_weekly_sales.mean(axis=0).round(1))

# Weekly totals must add up to the yearly totals computed directly from daily data
print("Check:", np.array_equal(total_weekly_sales.sum(axis=0), daily_sales.sum(axis=0)))

**Explanation.** The reshape groups every 7 consecutive days into a week, and `sum(axis=1)` adds them up, so `total_weekly_sales[0]` is the first week's total for each of the 4 products. The check confirms no data was lost: summing the weekly totals reproduces the yearly totals from the original daily data. Average weekly totals are roughly 365–390 units per product (7 days × ~55 units).

## Summary

- **Task 1:** simulated a `(364, 4)` array of daily sales with a fixed seed.
- **Task 2:** reshaped to `(12, 30, 4)` and averaged over days to get `(12, 4)` monthly averages.
- **Task 3:** showed that `np.sum` gives the same result as nested loops but is much faster.
- **Task 4:** reshaped to `(52, 7, 4)` and summed over days to get `(52, 4)` weekly totals.